In [0]:
# Define new target catalog/schema/volume
target_catalog = "landing_t"
target_schema = "landing_dlt_cdc"
target_volume = "staging"

spark.sql(f"CREATE CATALOG IF NOT EXISTS `{target_catalog}`")
spark.sql(f"USE CATALOG `{target_catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{target_catalog}`.`{target_schema}`")
spark.sql(f"USE SCHEMA `{target_schema}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{target_catalog}`.`{target_schema}`.`{target_volume}`")

# Path to the volume
target_volume_path = f"/Volumes/{target_catalog}/{target_schema}/{target_volume}"
print("Volume path:", target_volume_path)


In [0]:
spark.sql("SHOW CATALOGS").show()

In [0]:
spark.sql("DROP CATALOG main CASCADE")

In [0]:
spark.sql("SHOW CATALOGS").show()

In [0]:
# -----------------------------
# Configuration
# -----------------------------
source_catalog = "landing"
source_schema  = "ecommerce"
tables_to_copy = ["customers_base_metrics", "customers_demographics", "event_triggers","invoice_customer_bridge", "invoice_items", "journey_events", "marketing_campaigns", "marketing_spend", "products_dim"]  # Add all your tables here

target_catalog = "landing_t"
target_schema  = "landing_dlt_cdc"
target_volume  = "staging"

# Path to the target volume
target_volume_path = f"/Volumes/{target_catalog}/{target_schema}/{target_volume}"

# -----------------------------
# Loop over tables and copy them
# -----------------------------
for table_name in tables_to_copy:
    # Read the source table
    df = spark.table(f"{source_catalog}.{source_schema}.{table_name}")
    
    # Define target folder inside the volume
    target_folder = f"{target_volume_path}/{table_name}"
    
    # Write the table as JSON for raw ingestion
    df.repartition(50).write.format("json").mode("overwrite").save(target_folder)
    
    print(f"Copied {table_name} to {target_folder}")
